# 04 - Content-Based Filtering

This notebook builds and evaluates content-based movie representations:
1. TF-IDF features from genre tags
2. Genome tag features (1,128 dense relevance scores per movie)
3. Combined content similarity
4. Genre-only vs genome-enhanced comparison
5. Content-based recommendations for sample movies

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import json
import numpy as np
import pandas as pd
import scipy.sparse as sp
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid", palette="viridis")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["figure.dpi"] = 100

RAW_DIR = Path("../data/raw")
PROC_DIR = Path("../data/processed")

# Params
TFIDF_MAX_FEATURES = 5000
GENOME_WEIGHT = 0.7
GENRE_WEIGHT = 0.3

## 1. Build TF-IDF Features from Genres + Tags

Combine genre labels and user-applied tags into a text corpus, then apply TF-IDF.

In [ ]:
movies = pd.read_csv(RAW_DIR / "movies.csv")
tags = pd.read_csv(RAW_DIR / "tags.csv")

# Aggregate tags per movie
tag_agg = tags.groupby("movieId")["tag"].apply(lambda x: " ".join(x.dropna().astype(str))).reset_index()
tag_agg.columns = ["movieId", "tag_text"]

# Combine genres + tags into a single text field
movies_text = movies.copy()
movies_text["genre_text"] = movies_text["genres"].str.replace("|", " ", regex=False)
movies_text = movies_text.merge(tag_agg, on="movieId", how="left")
movies_text["tag_text"] = movies_text["tag_text"].fillna("")
movies_text["combined_text"] = movies_text["genre_text"] + " " + movies_text["tag_text"]

print(f"Movies with text: {len(movies_text)}")
print(f"Movies with tags: {movies_text['tag_text'].str.len().gt(0).sum()}")
display(movies_text[["movieId", "title", "combined_text"]].head())

In [ ]:
# TF-IDF
tfidf = TfidfVectorizer(max_features=TFIDF_MAX_FEATURES, stop_words="english")
tfidf_matrix = tfidf.fit_transform(movies_text["combined_text"])

print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")
print(f"Non-zero entries: {tfidf_matrix.nnz:,}")
print(f"Density: {tfidf_matrix.nnz / (tfidf_matrix.shape[0] * tfidf_matrix.shape[1]):.4%}")

# Top features
feature_names = tfidf.get_feature_names_out()
avg_tfidf = np.array(tfidf_matrix.mean(axis=0)).flatten()
top_feat_idx = np.argsort(avg_tfidf)[::-1][:20]

print("\nTop 20 TF-IDF features (by avg weight):")
for idx in top_feat_idx:
    print(f"  {feature_names[idx]:20s}  avg_weight={avg_tfidf[idx]:.4f}")

## 2. Build Genome Features

The genome scores provide a 1,128-dimensional relevance vector for each movie,
capturing nuanced attributes like "atmospheric", "thought-provoking", etc.

In [ ]:
genome_scores = pd.read_csv(RAW_DIR / "genome-scores.csv")
genome_tags = pd.read_csv(RAW_DIR / "genome-tags.csv")

print(f"Genome scores: {genome_scores.shape}")
print(f"Genome tags: {genome_tags.shape}")

# Pivot to movie x tag matrix
genome_pivot = genome_scores.pivot(index="movieId", columns="tagId", values="relevance")
genome_pivot = genome_pivot.fillna(0)

print(f"\nGenome feature matrix: {genome_pivot.shape}")
print(f"Movies with genome data: {len(genome_pivot)}")
print(f"Movies in catalog: {len(movies)}")
print(f"Coverage: {len(genome_pivot) / len(movies) * 100:.1f}%")

In [ ]:
# Visualise genome features for a sample movie (Toy Story)
toy_story_id = movies[movies["title"].str.contains("Toy Story") & movies["title"].str.contains("1995")]["movieId"].values

if len(toy_story_id) > 0 and toy_story_id[0] in genome_pivot.index:
    ts_genome = genome_pivot.loc[toy_story_id[0]]
    top_tag_ids = ts_genome.nlargest(20).index
    top_tag_names = genome_tags.set_index("tagId").loc[top_tag_ids, "tag"].values
    top_tag_values = ts_genome[top_tag_ids].values
    
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.barh(range(len(top_tag_names)), top_tag_values[::-1], color="steelblue", edgecolor="white")
    ax.set_yticks(range(len(top_tag_names)))
    ax.set_yticklabels(top_tag_names[::-1])
    ax.set_xlabel("Genome Relevance Score")
    ax.set_title("Top 20 Genome Tags for Toy Story (1995)")
    plt.tight_layout()
    plt.show()
else:
    print("Toy Story not found in genome data.")

## 3. Content Similarity Examples

Compute cosine similarity using the combined content features and find similar movies.

In [ ]:
# Align TF-IDF and genome features to the same movie set
tfidf_movie_ids = movies_text["movieId"].values
genome_movie_ids = genome_pivot.index.values
common_movies = np.intersect1d(tfidf_movie_ids, genome_movie_ids)
print(f"Movies in both TF-IDF and genome: {len(common_movies)}")

# Build aligned matrices
tfidf_id_to_idx = {mid: idx for idx, mid in enumerate(tfidf_movie_ids)}
genome_id_to_idx = {mid: idx for idx, mid in enumerate(genome_movie_ids)}

tfidf_aligned = tfidf_matrix[[tfidf_id_to_idx[m] for m in common_movies]]
genome_aligned = genome_pivot.loc[common_movies].values

# Normalise both
tfidf_normed = normalize(tfidf_aligned, norm="l2")
genome_normed = normalize(genome_aligned, norm="l2")

# Weighted combination
combined = sp.hstack([
    tfidf_normed * GENRE_WEIGHT,
    sp.csr_matrix(genome_normed) * GENOME_WEIGHT,
])
combined = normalize(combined, norm="l2")
print(f"Combined content matrix: {combined.shape}")

In [ ]:
# Find movies similar to Toy Story using combined features
movie_title_map = dict(zip(movies["movieId"], movies["title"]))
common_id_to_idx = {mid: idx for idx, mid in enumerate(common_movies)}

def find_similar(movie_id, feature_matrix, top_n=10):
    """Find top-N similar movies by cosine similarity."""
    if movie_id not in common_id_to_idx:
        return []
    idx = common_id_to_idx[movie_id]
    sims = cosine_similarity(feature_matrix[idx], feature_matrix).flatten()
    sims[idx] = 0  # exclude self
    top_idx = np.argsort(sims)[::-1][:top_n]
    return [(common_movies[i], movie_title_map.get(common_movies[i], "?"), sims[i]) for i in top_idx]

# Similar to Toy Story
if len(toy_story_id) > 0:
    print("Movies similar to Toy Story (combined content features):")
    print("-" * 65)
    for mid, title, score in find_similar(toy_story_id[0], combined, top_n=15):
        print(f"  {title:50s}  sim={score:.4f}")

In [ ]:
# More examples: The Dark Knight, Pulp Fiction
example_movies = {
    "The Dark Knight": movies[movies["title"].str.contains("Dark Knight") & movies["title"].str.contains("2008")]["movieId"].values,
    "Pulp Fiction": movies[movies["title"].str.contains("Pulp Fiction")]["movieId"].values,
}

for name, ids in example_movies.items():
    if len(ids) > 0:
        print(f"\nMovies similar to {name}:")
        print("-" * 65)
        for mid, title, score in find_similar(ids[0], combined, top_n=10):
            print(f"  {title:50s}  sim={score:.4f}")

## 4. Genre-Only vs Genome-Enhanced Comparison

Compare similarity quality with genre-only TF-IDF vs genome-enhanced features.

In [ ]:
if len(toy_story_id) > 0 and toy_story_id[0] in common_id_to_idx:
    genre_only_similar = find_similar(toy_story_id[0], tfidf_normed, top_n=10)
    genome_enhanced_similar = find_similar(toy_story_id[0], combined, top_n=10)
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # Genre-only
    titles_g = [t[:35] for _, t, _ in genre_only_similar]
    scores_g = [s for _, _, s in genre_only_similar]
    axes[0].barh(range(len(titles_g)), scores_g[::-1], color="#FF9800", edgecolor="white")
    axes[0].set_yticks(range(len(titles_g)))
    axes[0].set_yticklabels(titles_g[::-1], fontsize=9)
    axes[0].set_xlabel("Cosine Similarity")
    axes[0].set_title("Genre-Only (TF-IDF)")
    
    # Genome-enhanced
    titles_c = [t[:35] for _, t, _ in genome_enhanced_similar]
    scores_c = [s for _, _, s in genome_enhanced_similar]
    axes[1].barh(range(len(titles_c)), scores_c[::-1], color="steelblue", edgecolor="white")
    axes[1].set_yticks(range(len(titles_c)))
    axes[1].set_yticklabels(titles_c[::-1], fontsize=9)
    axes[1].set_xlabel("Cosine Similarity")
    axes[1].set_title("Genome-Enhanced (Combined)")
    
    plt.suptitle("Similar to Toy Story: Genre-Only vs Genome-Enhanced", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()
    
    # Overlap analysis
    genre_set = {mid for mid, _, _ in genre_only_similar}
    combined_set = {mid for mid, _, _ in genome_enhanced_similar}
    overlap = genre_set & combined_set
    print(f"\nOverlap between top-10 lists: {len(overlap)} / 10")
    print("Genome features add nuance beyond genre categorisation.")
else:
    print("Toy Story not available for comparison.")

In [ ]:
# Distribution comparison of similarity values
sample_idx = np.random.RandomState(42).choice(len(common_movies), size=500, replace=False)
genre_sims = cosine_similarity(tfidf_normed[sample_idx]).flatten()
combined_sims = cosine_similarity(combined[sample_idx]).flatten()

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(genre_sims[genre_sims > 0.01], bins=50, alpha=0.6, color="#FF9800", label="Genre-Only", density=True)
ax.hist(combined_sims[combined_sims > 0.01], bins=50, alpha=0.6, color="steelblue", label="Genome-Enhanced", density=True)
ax.set_xlabel("Cosine Similarity")
ax.set_ylabel("Density")
ax.set_title("Similarity Score Distribution: Genre-Only vs Genome-Enhanced")
ax.legend()
plt.tight_layout()
plt.show()

print("Content-based filtering exploration complete.")